In [11]:
import cv2
import glob
import numpy as np
import rawpy

In [12]:
# find all sections
sampleID = sorted(glob.glob("raw_data/*"))
print(sampleID)

['raw_data/ALHIC2502-102-A', 'raw_data/ALHIC2502-102-C', 'raw_data/ALHIC2502-102-D', 'raw_data/ALHIC2502-103', 'raw_data/ALHIC2502-24', 'raw_data/ALHIC2502-47', 'raw_data/ALHIC2502-86']


In [13]:
# set params
scale_factor = 0.15  # Downsample to 10% size for feature matching

In [ ]:
# loop through folders
for sample in sampleID:

    print("Running images from " + sample)

    # find files
    nef_paths = sorted(glob.glob(f"{sample}/*.NEF"))
    print(f"    Found {len(nef_paths)} NEF files in {sample}.")

    # downscale for matching
    print("    Step 1: Extracting low-res thumbnails to calculate alignment...")
    low_res_images = []
    for path in nef_paths:
        # Process one RAW file at a time, immediately freeing memory
        with rawpy.imread(path) as raw:
            # half_size=True fast-decodes a smaller preview, saving CPU and RAM
            rgb = raw.postprocess(use_camera_wb=True, half_size=True)
            
        # Convert RGB (rawpy default) to BGR (OpenCV default)
        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        
        # Resize heavily for the stitcher calculation
        h, w = bgr.shape[:2]
        target_w = int(w * scale_factor * 2) # Adjusting for half_size decode
        target_h = int(h * scale_factor * 2)
        
        small_img = cv2.resize(bgr, (target_w, target_h), interpolation=cv2.INTER_AREA)
        low_res_images.append(small_img)

    # 2. Compute mapping using SCANS mode
    print("    Step 2: Aligning 2D zig-zag grid structure...")

    # CRITICAL CHANGE: cv2.STITCHER_MODE_SCANS is built for flat flat X/Y translation grids
    #stitcher = cv2.Stitcher_create(cv2.Stitcher_MODE_SCANS) 
    stitcher = cv2.Stitcher.create(cv2.Stitcher_SCANS)

    status, low_res_panorama = stitcher.stitch(low_res_images)

    if status != cv2.Stitcher_OK:
        print(f"Stitching failed! Error code: {status}")
        print("Tip: In a zig-zag pattern, check the 'corners' where it moves up.")
        print("If there isn't enough vertical overlap when stepping up, it will break.")
        exit()

    print("Grid layout successfully calculated!")

    # 3. Apply to full-resolution images sequentially
    print("    Step 3: Generating final high-res grid composition...")
    stitcher_high = cv2.Stitcher_create(cv2.Stitcher_MODE_SCANS)
    stitcher_high.setRegistrationResol(0.6)  # Keeps memory usage locked low

    high_res_images = []
    for path in nef_paths:
        with rawpy.imread(path) as raw:
            rgb = raw.postprocess(use_camera_wb=True, half_size=False)
        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        high_res_images.append(bgr)

    status, high_res_panorama = stitcher_high.stitch(high_res_images)

    if status == cv2.Stitcher_OK:
        output_path = "ice_core_grid_stitched.jpg"
        cv2.imwrite(output_path, high_res_panorama)
        print(f"    Success! Stitched grid saved to: {output_path}")
    else:
        print(f"High resolution layout processing failed: {status}")

    # 3. Apply to full-resolution images sequentially
    print("    Step 3: Generating final high-res grid composition...")
    stitcher_high = cv2.Stitcher_create(cv2.Stitcher_SCANS)
    stitcher_high.setRegistrationResol(0.6)  # Keeps memory usage locked low

    high_res_images = []
    for path in nef_paths:
        with rawpy.imread(path) as raw:
            rgb = raw.postprocess(use_camera_wb=True, half_size=False)
        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        high_res_images.append(bgr)

        # simple text progress bar
        progress = len(high_res_images) / len(nef_paths)
        bar_length = 30
        filled = int(bar_length * progress)
        bar = "#" * filled + "-" * (bar_length - filled)
        print(
            f"\r    [{bar}] {len(high_res_images)}/{len(nef_paths)} "
            f"({progress * 100:5.1f}%)",
            end="",
            flush=True,
        )
        if len(high_res_images) == len(nef_paths):
            print()

    status, high_res_panorama = stitcher_high.stitch(high_res_images)

    if status == cv2.Stitcher_OK:
        output_path = "image_outputs"+sample+".jpg"
        cv2.imwrite(output_path, high_res_panorama)
        print(f"\nSuccess! Stitched grid saved to: {output_path}")
    else:
        print(f"High resolution layout processing failed: {status}")


Running images from raw_data/ALHIC2502-102-A
    Found 49 NEF files in raw_data/ALHIC2502-102-A.
Step 1: Extracting low-res thumbnails to calculate alignment...

Step 2: Aligning 2D zig-zag grid structure...
Grid layout successfully calculated!

Step 3: Generating final high-res grid composition...


AttributeError: module 'cv2' has no attribute 'Stitcher_MODE_SCANS'